# 곡 구간 → 연습 줄 확인·수정 (이슈 #31)

내 노래의 한 구간을 **한 줄(문장) 단위**로 나눈 결과를 **귀로 확인**하고, STT가 틀린 가사를 **고쳐서** 연습 문장으로 저장합니다. 작업 목록과 설계는 `작업기록` 문서(`곡_구간_연습문장_만들기_작업기록.md`)에 있습니다.

**이 노트북에서 하는 일 (순서대로)**
1. **구간 선택**: `CLIP_ID`를 정합니다(오늘은 `c02` → `c13` → `c01` 순서를 권합니다).
2. **줄 후보 불러오기**: 저장된 줄 목록을 읽습니다(없으면 `medium`으로 인식해 만듭니다. 구간당 30~60초).
3. **줄별 재생으로 귀 확인**(E1): 각 줄을 들으며 **경계가 맞는지** 봅니다.
4. **경계 메모**(선택): 이상한 줄 번호와 짧은 메모를 남깁니다(줄 나누기 규칙을 고칠 때 근거가 됩니다).
5. **가사 수정**(E2): 소리와 다른 글을 고치거나, 맞다고 확인하거나, 연습에서 뺍니다. **`APPLY = True`로 바꿔야 저장됩니다.**
6. **표준 발음 기준 음성 계획**: 고친 가사를 OpenAI TTS로 만들 때 **몇 번 호출(과금)하는지 먼저** 봅니다. `GENERATE = True`일 때만 실제로 호출합니다.

**주의**
- 이 노트북의 **출력에는 곡의 소리(오디오 플레이어)와 가사가 들어갑니다.** 저장소에 올리면 안 되므로 **커밋 전에 출력을 지운 정리본만** 올립니다(`scripts/scan_outputs.py`로 확인).
- 줄 목록과 수정한 가사는 `data/song_practice/`(git 제외)에만 저장됩니다. **고친 가사는 다시 만들 때 덮어써지지 않습니다.**
- **표시된 "인식 신뢰가 낮을 수 있음"은 클립 단위 힌트**입니다(줄마다 다른 값이 아님). 그 구간은 글을 더 꼼꼼히 확인하세요.

In [ ]:
# ① 설정: 프로젝트 루트로 이동하고 모듈을 불러옵니다. (이 노트북은 experiments/ 아래에 있어서 상대 경로 data/...가 맞도록 루트로 옮깁니다.)
import os
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "song_practice.py").exists() and root != root.parent:
    root = root.parent                      # 위 폴더로 올라가며 프로젝트 루트(song_practice.py가 있는 곳)를 찾는다
os.chdir(root)
sys.path.insert(0, str(root))

try:                                        # .py를 고치면 커널을 재시작하지 않아도 다시 읽게 한다(노트북에서만 동작)
    ip = get_ipython()
    ip.run_line_magic("load_ext", "autoreload")
    ip.run_line_magic("autoreload", "2")
except Exception:
    pass

import pandas as pd
from IPython.display import Audio, display

import audio            # 파일 읽기(원본 샘플)
import separate         # 구간 파일 경로 규칙(clip_paths)·구간 목록(clips.csv)
import song_practice    # 줄 목록 만들기·저장·수정

print("프로젝트 루트:", root.name)

## ② 구간 선택
`CLIP_ID`만 바꿔서 다시 실행하면 다른 구간으로 넘어갑니다. 구간 파일은 `data/song_work/<id>_mix.wav`(원곡)입니다.

In [ ]:
CLIP_ID = "c02"          # 오늘 권하는 순서: c02 → c13 → c01 (모두 인식이 정확했던 구간이라 가사 수정이 가볍습니다)

meta = separate.manifest_row(CLIP_ID) or {}
mix_path, _ = separate.clip_paths(CLIP_ID)
print(f"구간 {CLIP_ID}: 곡 안 {meta.get('start')}초부터 {meta.get('duration')}초 | 구간 파일 있음: {mix_path.exists()} | 저장된 줄 목록 있음: {song_practice.lines_path(CLIP_ID).exists()}")

## ③ 줄 후보 불러오기
저장된 `lines.json`이 있으면 그대로 읽고, **없을 때만** `medium`으로 인식해서 만듭니다(처음 한 번 30~60초, 모델 읽기 포함). 이미 있는 파일은 **덮어쓰지 않습니다.**

> **이 글은 어디서 오나?** `lines.json`의 `stt_text`와 `text`는 모두 **모델(`medium`)이 원곡 구간을 인식한 글**입니다. 제가 직접 받아 적은 정답 파일(`data/song_work/truth/<id>.txt`)이 **아닙니다.** 정답 파일은 인식 실험의 점수 계산(`clip_compare.py`)에만 쓰이고 `song_practice.py`는 읽지 않습니다.
> - `stt_text`는 인식 원문이고 항상 그대로 남습니다. `text`는 처음에는 `stt_text`와 같고, ⑦에서 고치면 바뀝니다(`edited=True`).
> - 그래서 **고치지 않고 확정한 줄은 모델이 인식한 글이 그대로 연습 문장이 됩니다.** 모델이 틀린 단어도 그대로 TTS로 읽힙니다.
> - c01·c02·c13의 글이 정답과 거의 같은 것은(정답 대비 오류 1·0·2단어) 정답에서 가져와서가 아니라 **모델이 이 구간들을 거의 정확히 인식했기 때문**입니다.

In [ ]:
if song_practice.lines_path(CLIP_ID).exists():
    doc = song_practice.load_lines(CLIP_ID)
else:
    doc = song_practice.prepare_lines(CLIP_ID)      # medium + 원곡 경로 + VAD 끔 + 기본 설정, 줄 나누기까지
print("STT 설정:", doc["stt"])
print("줄 수:", len(doc["lines"]), "| 상태별:", song_practice.status_summary(CLIP_ID))

## ④ 줄 후보 표 (가사 포함)
`출처`는 이 줄이 어떻게 만들어졌는지입니다: `segment`(Whisper 구간 그대로), `split`(긴 구간을 단어 사이 쉼에서 쪼갬), `merged`(짧은 구간을 이웃과 합침). `STT 글`이 **고칠 대상**입니다.

In [ ]:
rows = [{"줄": l["no"], "시작(초)": l["start"], "끝(초)": l["end"], "길이": round(l["end"] - l["start"], 1), "단어": l["n_words"],
         "출처": l["source"], "상태": l["status"], "STT 글": l["text"]} for l in doc["lines"]]
pd.set_option("display.max_colwidth", 120)
display(pd.DataFrame(rows).set_index("줄"))
if any(l["low_confidence"] for l in doc["lines"]):
    print("※ 이 구간은 인식 신뢰가 낮을 수 있는 소리입니다(클립 단위 힌트). 글을 더 꼼꼼히 확인하세요.")

## ⑤ 줄별 재생으로 귀 확인 (E1)
각 줄 앞뒤로 `PAD`초씩 여유를 두고 재생합니다. **들으면서 확인할 것**
- 줄의 **시작과 끝**이 가사의 줄(숨 쉬는 곳)과 맞는가?
- 한 줄 안에 **두 줄이 섞였거나**, 한 줄이 **둘로 잘렸는가**?
- 아래 `STT 글`이 소리와 같은가? (다르면 ⑦에서 고칩니다)

In [ ]:
PAD = 0.3                                   # 줄 앞뒤로 더 들려줄 초(경계 소리가 잘리지 않게)
x, sr, _ch = audio.load_raw(mix_path)
mono = x.mean(axis=1)                       # 재생용 모노(스테레오를 평균), 원곡 그대로의 표본화 주파수


def part(start, end):
    return mono[max(int((start - PAD) * sr), 0): int((end + PAD) * sr)]


print("구간 전체")
display(Audio(mono, rate=sr))
for l in doc["lines"]:
    print(f"줄 {l['no']}  {l['start']:.1f}~{l['end']:.1f}초 ({l['source']}, {l['status']})  |  {l['text']}")
    display(Audio(part(l["start"], l["end"]), rate=sr))

## ⑥ 경계 메모 (선택)
이상한 줄이 있으면 **줄 번호와 짧은 메모**를 적어 두세요. 줄 나누기 규칙(`song_lines.py`)을 고칠 때 근거가 됩니다. 가사는 바뀌지 않습니다. 모두 괜찮으면 비워 두세요.

In [ ]:
BOUNDARY_NOTES = {}        # 예: {2: "너무 짧음", 4: "앞 줄과 합쳐야 함", 5: "한 줄이 둘로 잘림"}
SAVE_NOTES = False         # True로 바꾸고 실행하면 lines.json에 저장

if SAVE_NOTES:
    print("저장함:", song_practice.save_review(CLIP_ID, BOUNDARY_NOTES))
else:
    print("SAVE_NOTES=False → 저장하지 않았습니다. 메모:", BOUNDARY_NOTES or "없음")

## ⑦ 가사 수정 (E2)
- `EDITS`: **소리와 다른 줄**을 `{줄 번호: "고친 가사"}`로 적습니다. 고친 줄은 자동으로 **확정**됩니다(수정 전 STT 글은 따로 남습니다).
- `CONFIRM`: **고치지 않고 맞다고 확인한 줄** 번호입니다.
- `DROP`: **연습에서 뺄 줄**(반주만 있는 구간, 추임새 등) 번호입니다.
- 마지막에 `APPLY = True`로 바꿔 실행하면 **저장**됩니다. 틀린 번호나 빈 가사가 있으면 **아무것도 저장하지 않고** 오류를 냅니다.

In [ ]:
EDITS = {}        # 예: {2: "고친 가사 …"}
CONFIRM = []      # 예: [1, 3, 4]  (고치지 않고 맞다고 확인한 줄)
DROP = []         # 예: [5]
APPLY = False     # True로 바꾸고 실행하면 저장

if APPLY:
    doc = song_practice.apply_edits(CLIP_ID, EDITS, CONFIRM, DROP)
    print("저장함:", song_practice.status_summary(CLIP_ID))
else:
    print(f"APPLY=False → 저장하지 않았습니다. 적용 예정: 고침 {sorted(EDITS)}, 확인 {sorted(CONFIRM)}, 제외 {sorted(DROP)}")

## ⑧ 수정 전/후 비교
저장된 줄 중 **고친 줄**의 STT 원문과 수정본을 나란히 보여 줍니다(수정 전/후가 모두 `lines.json`에 남아 있습니다).

In [ ]:
doc = song_practice.load_lines(CLIP_ID)
print("상태별:", song_practice.status_summary(CLIP_ID))
edited = [{"줄": l["no"], "STT 원문": l["stt_text"], "수정본": l["text"]} for l in doc["lines"] if l["edited"]]
display(pd.DataFrame(edited).set_index("줄")) if edited else print("고친 줄이 아직 없습니다.")

## ⑨ 표준 발음 기준 음성 (OpenAI TTS)
**확정된 줄**의 수정 가사를 OpenAI TTS가 읽은 소리를 줄마다 만듭니다. **호출은 과금**됩니다. 가사를 외부 API로 보내는 것은 11-04에 허용했습니다(검토 문서 13.5).
- `GENERATE = False`(기본): **호출 없이 계획만** 보여 줍니다(새로 부를 횟수, 캐시에 이미 있어 부르지 않는 줄 수, 보낼 글자 수).
- `GENERATE = True`: 실제로 만들고 `data/`에 저장한 뒤 줄마다 재생합니다. 같은 문장은 캐시를 써서 다시 부르지 않습니다.

In [ ]:
GENERATE = False         # 먼저 False로 호출 횟수를 확인하세요. True면 OpenAI TTS를 호출합니다(과금).

plan = song_practice.make_tts(CLIP_ID, generate=GENERATE)
print("계획:", plan)
if plan["done"]:
    doc = song_practice.load_lines(CLIP_ID)
    for l in doc["lines"]:
        if l["status"] == "confirmed" and l.get("tts_path"):
            print(f"줄 {l['no']}  {l['text']}")
            display(Audio(filename=l["tts_path"]))

## 끝낸 뒤
- 이 노트북을 커밋할 때는 **출력을 지운 정리본**만 올립니다(곡의 소리와 가사가 출력에 들어 있습니다). 로컬 원본은 그대로 둡니다.
- 확인 결과(경계가 맞았는지, 몇 줄을 고쳤는지)는 **숫자로만** 작업기록 문서에 적습니다. 가사는 옮겨 적지 않습니다.